In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

!pip uninstall -y torch torchaudio torchvision vllm

!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -q --no-index --find-links=/kaggle/input/vllm-wheels-cache vllm

!pip install -q sentence-transformers transformers accelerate

In [ ]:

!pip uninstall -y torch torchaudio torchvision

!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129

In [ ]:
!pip install vllm==0.11.0

In [ ]:
!pip install "transformers>=4.40.0,<5.0.0"

In [ ]:
import torch
import torchaudio

print(f"PyTorch: {torch.__version__}")
print(f"TorchAudio: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and "awq" in root.lower():
        print(root)
        break

In [2]:
!pip uninstall -y vllm torch torchaudio torchvision transformers
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129
!pip install vllm==0.11.0
!pip install "transformers>=4.40.0,<5.0.0"

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
Looking in indexes: https://download.pytorch.org/whl/cu129
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 14.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 139.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 108.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 66.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [4]:
import torch
import torchaudio
import vllm
import transformers

print(f"torch: {torch.__version__}")
print(f"torchaudio: {torchaudio.__version__}")
print(f"vllm: {vllm.__version__}")
print(f"transformers: {transformers.__version__}")

torch: 2.8.0+cu129
torchaudio: 2.8.0+cu129
vllm: 0.11.0
transformers: 4.57.6


In [5]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/foroughkoohi/graphrag-qa-data")
DST = Path("/kaggle/working")

for item in SRC.iterdir():
    dst = DST / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy(item, dst)

!ls -la /kaggle/working

total 32
drwxr-xr-x 6 root root 4096 Oct 14 19:10 .
drwxr-xr-x 5 root root 4096 Oct 14 18:59 ..
drwxr-xr-x 3 root root 4096 Oct 14 10:23 cache
drwxr-xr-x 3 root root 4096 Oct 14 10:24 data
-rw-r--r-- 1 root root 8120 Oct 14 19:10 score.py
drwxr-xr-x 2 root root 4096 Oct 14 10:28 src
drwxr-xr-x 2 root root 4096 Oct 14 19:00 .virtual_documents


In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
OUT_DIR = Path("/kaggle/working/runs/hidden/closed_book")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
)
sampling = SamplingParams(temperature=0, max_tokens=30)

PROMPT_TEMPLATE = """Answer the question with a SHORT answer.
Rules:
- Use lowercase only
- No punctuation at the end
- For yes/no questions, answer exactly 'yes' or 'no'
- Do not explain
- Just give the answer

Question: {question}
Answer:"""

questions = [json.loads(l) for l in open(PACK / "questions_hidden.jsonl", encoding="utf-8")]
print(f"Number of questions: {len(questions)}")

prompts = [PROMPT_TEMPLATE.format(question=q["question"]) for q in questions]

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"{time.time()-t0:.1f} seconds")

with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, out in zip(questions, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": [],
        }, ensure_ascii=False) + "\n")

print("Saved")

In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
OUT_DIR = Path("/kaggle/working/runs/hidden/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.85,   
    max_num_seqs=8,                 
    max_num_batched_tokens=4096,    
)
sampling = SamplingParams(temperature=0, max_tokens=30)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"{time.time()-t0:.1f} seconds")

# Save
OUT_DIR = Path("/kaggle/working/runs/hidden/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, rids, out in zip(questions, retrieved_ids, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": rids,
        }, ensure_ascii=False) + "\n")

print("Saved")

In [3]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,  # faster
)

sampling = SamplingParams(temperature=0, max_tokens=30)

# ===== CLOSED-BOOK on dev =====
PROMPT_TEMPLATE = """Answer the question with a SHORT answer.
Rules:
- Use lowercase only
- No punctuation at the end
- For yes/no questions, answer exactly 'yes' or 'no'
- Do not explain
- Just give the answer

Question: {question}
Answer:"""

questions = [json.loads(l) for l in open(PACK / "questions_dev.jsonl", encoding="utf-8")]
prompts = [PROMPT_TEMPLATE.format(question=q["question"]) for q in questions]

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"closed-book: {time.time()-t0:.1f} seconds")

OUT_DIR = Path("/kaggle/working/runs/dev/closed_book")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, out in zip(questions, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": [],
        }, ensure_ascii=False) + "\n")

print("closed-book saved")

# ===== DENSE on dev =====
import numpy as np
from sentence_transformers import SentenceTransformer

INDEX_DIR = Path("/kaggle/working/cache/index")
embeddings = np.load(INDEX_DIR / "embeddings.npy")
with open(INDEX_DIR / "doc_ids.json") as f:
    doc_ids = json.load(f)

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
corpus = {p["doc_id"]: p["text"] for p in [json.loads(l) for l in open(PACK / "corpus.jsonl", encoding="utf-8")]}

q_texts = [q["question"] for q in questions]
q_embs = embedder.encode(q_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

scores = q_embs @ embeddings.T
top_idx = np.argsort(-scores, axis=1)[:, :5]
retrieved_ids = [[doc_ids[i] for i in row] for row in top_idx]

RAG_PROMPT = """Answer the question based on the context below.
Rules:
- Use lowercase only
- No punctuation at the end
- For yes/no questions, answer exactly 'yes' or 'no'
- Do not explain
- Just give the answer

Context:
{context}

Question: {question}
Answer:"""

prompts = []
for q, rids in zip(questions, retrieved_ids):
    ctx = "\n\n".join(f"- {corpus[did]}" for did in rids)
    prompts.append(RAG_PROMPT.format(context=ctx, question=q["question"]))

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"dense: {time.time()-t0:.1f} seconds")

OUT_DIR = Path("/kaggle/working/runs/dev/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, rids, out in zip(questions, retrieved_ids, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": rids,
        }, ensure_ascii=False) + "\n")

print("dense saved")

# ===== Scoring =====
!python /kaggle/working/score.py /kaggle/working/runs/dev/closed_book/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

!python /kaggle/working/score.py /kaggle/working/runs/dev/dense/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

INFO 10-14 17:56:09 [__init__.py:216] Automatically detected platform cuda.
INFO 10-14 17:56:31 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-14 17:56:51 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-14 17:56:51 [model.py:1510] Using max model len 4096
INFO 10-14 17:56:52 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-14 17:56:52 [__init__.py:381] Cudagraph is disabled under eager mode
WARNING 10-14 17:56:53 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 10-14 17:57:02 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=483) INFO 10-14 17:57:05 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=483) INFO 10-14 17:57:06 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', skip_

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:54<00:54, 54.00s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [01:10<00:00, 32.09s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [01:10<00:00, 35.38s/it]
(EngineCore_DP0 pid=483) 


(EngineCore_DP0 pid=483) INFO 10-14 17:58:22 [default_loader.py:267] Loading weights took 70.77 seconds
(EngineCore_DP0 pid=483) INFO 10-14 17:58:23 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 71.028533 seconds
(EngineCore_DP0 pid=483) INFO 10-14 17:58:30 [gpu_worker.py:298] Available KV cache memory: 5.92 GiB
(EngineCore_DP0 pid=483) INFO 10-14 17:58:30 [kv_cache_utils.py:1087] GPU KV cache size: 43,072 tokens
(EngineCore_DP0 pid=483) INFO 10-14 17:58:30 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.52x
(EngineCore_DP0 pid=483) WARNING 10-14 17:58:30 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=483) INFO 10-14 17:58:31 [core.py:210] init engine (profile, create kv cache, warmup model) took 7.76 seconds
(EngineCore_DP0 pid=483) INFO 10-14 17:58:31 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 17:58:32 [llm.py:306] Supported_tasks: ['gene

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

dense: 891.4 seconds
dense saved
valid: /kaggle/working/runs/dev/closed_book/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                18.0   [ 14.0,  22.7]
  f1                21.3   [ 17.2,  26.0]

type: bridge_comparison (n=66)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                24.2   [ 13.6,  34.8]
  f1                24.2   [ 13.6,  34.8

In [9]:
import gc
import torch

# Free the previous llm
try:
    del llm
    print("Old llm deleted")
except NameError:
    print("llm was not defined")

# Garbage collection
gc.collect()
torch.cuda.empty_cache()

# Check free memory
print(f"Free memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total memory: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

[rank0]:[W1014 18:37:15.394176658 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Old llm deleted
Free memory: 15.33 GB
Total memory: 15.64 GB


In [7]:
# Oracle on dev (300 questions)
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)

sampling = SamplingParams(temperature=0, max_tokens=30)

questions = [json.loads(l) for l in open(PACK / "questions_dev.jsonl", encoding="utf-8")]
labels = {l["qid"]: l for l in [json.loads(l) for l in open(PACK / "labels_dev.jsonl", encoding="utf-8")]}
corpus = {p["doc_id"]: p["text"] for p in [json.loads(l) for l in open(PACK / "corpus.jsonl", encoding="utf-8")]}

PROMPT = """Answer the question based on the context below.
Rules:
- Use lowercase only
- No punctuation at the end
- For yes/no questions, answer exactly 'yes' or 'no'
- Do not explain

Context:
{context}

Question: {question}
Answer:"""

prompts = []
for q in questions:
    gold_ids = labels[q["qid"]]["gold_doc_ids"]
    ctx = "\n\n".join(f"- {corpus[did]}" for did in gold_ids)
    prompts.append(PROMPT.format(context=ctx, question=q["question"]))

import time
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"oracle: {time.time()-t0:.1f} seconds")

OUT_DIR = Path("/kaggle/working/runs/dev/oracle")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, out in zip(questions, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": labels[q["qid"]]["gold_doc_ids"],
        }, ensure_ascii=False) + "\n")

print("oracle saved")

# Score
!python /kaggle/working/score.py /kaggle/working/runs/dev/oracle/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

INFO 10-14 18:25:14 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-14 18:25:15 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 10-14 18:25:15 [model.py:1510] Using max model len 4096
INFO 10-14 18:25:15 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-14 18:25:15 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 18:25:24 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:27 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:27 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.36s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.30it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.17it/s]
(EngineCore_DP0 pid=1205) 


(EngineCore_DP0 pid=1205) INFO 10-14 18:25:33 [default_loader.py:267] Loading weights took 1.73 seconds
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:34 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 1.969255 seconds
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:42 [gpu_worker.py:298] Available KV cache memory: 5.92 GiB
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:43 [kv_cache_utils.py:1087] GPU KV cache size: 43,072 tokens
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:43 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.52x
(EngineCore_DP0 pid=1205) WARNING 10-14 18:25:43 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:43 [core.py:210] init engine (profile, create kv cache, warmup model) took 9.03 seconds
(EngineCore_DP0 pid=1205) INFO 10-14 18:25:44 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 18:25:44 [llm.py:306] Supported_tasks: 

Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

oracle: 598.6 seconds
oracle saved
valid: /kaggle/working/runs/dev/oracle/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          89.0   [ 86.5,  91.3]
  hitall@2          78.0   [ 73.0,  82.7]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                51.3   [ 46.0,  56.7]
  f1                63.3   [ 58.7,  68.3]

type: bridge_comparison (n=66)
  recall@2          50.0   [ 50.0,  50.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                48.5   [ 36.4,  60.6]
  f1                52.0   [ 40.3,  63.5]



In [10]:
# Oracle on dev (300 questions)
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)

sampling = SamplingParams(temperature=0, max_tokens=30)

questions = [json.loads(l) for l in open(PACK / "questions_dev.jsonl", encoding="utf-8")]
labels = {l["qid"]: l for l in [json.loads(l) for l in open(PACK / "labels_dev.jsonl", encoding="utf-8")]}
corpus = {p["doc_id"]: p["text"] for p in [json.loads(l) for l in open(PACK / "corpus.jsonl", encoding="utf-8")]}

PROMPT = """Answer the question with the SHORTEST possible answer.
CRITICAL RULES:
- If the answer is an entity, give ONLY the entity name
- If the answer is a date, give ONLY the date
- If the answer is a place, give ONLY the place name
- If it's a yes/no question, answer exactly 'yes' or 'no'
- DO NOT write sentences
- DO NOT explain
- DO NOT start with "based on" or "the context"
- Use lowercase
- No punctuation

Context:
{context}

Question: {question}
Answer:"""

prompts = []
for q in questions:
    gold_ids = labels[q["qid"]]["gold_doc_ids"]
    ctx = "\n\n".join(f"- {corpus[did]}" for did in gold_ids)
    prompts.append(PROMPT.format(context=ctx, question=q["question"]))

import time
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"oracle: {time.time()-t0:.1f} seconds")

OUT_DIR = Path("/kaggle/working/runs/dev/oracle")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, out in zip(questions, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": labels[q["qid"]]["gold_doc_ids"],
        }, ensure_ascii=False) + "\n")

print("oracle saved")

# Score
!python /kaggle/working/score.py /kaggle/working/runs/dev/oracle/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

INFO 10-14 18:37:29 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-14 18:37:29 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 10-14 18:37:29 [model.py:1510] Using max model len 4096
INFO 10-14 18:37:29 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-14 18:37:29 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 18:37:38 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:41 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:41 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.40s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.15it/s]
(EngineCore_DP0 pid=1527) 


(EngineCore_DP0 pid=1527) INFO 10-14 18:37:47 [default_loader.py:267] Loading weights took 1.75 seconds
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:48 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 2.008846 seconds
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:56 [gpu_worker.py:298] Available KV cache memory: 5.92 GiB
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:57 [kv_cache_utils.py:1087] GPU KV cache size: 43,072 tokens
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:57 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.52x
(EngineCore_DP0 pid=1527) WARNING 10-14 18:37:57 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:57 [core.py:210] init engine (profile, create kv cache, warmup model) took 8.67 seconds
(EngineCore_DP0 pid=1527) INFO 10-14 18:37:58 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 18:37:58 [llm.py:306] Supported_tasks: 

Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

oracle: 602.1 seconds
oracle saved
valid: /kaggle/working/runs/dev/oracle/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          89.0   [ 86.5,  91.3]
  hitall@2          78.0   [ 73.0,  82.7]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                61.3   [ 56.0,  66.3]
  f1                69.4   [ 64.8,  74.0]

type: bridge_comparison (n=66)
  recall@2          50.0   [ 50.0,  50.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                54.5   [ 42.4,  66.7]
  f1                55.1   [ 42.6,  67.2]



In [6]:
# Dense on dev (300 questions) - STRICT PROMPT
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams
import numpy as np
from sentence_transformers import SentenceTransformer

PACK = Path("/kaggle/working/data/course_pack_v1")
MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

# ===== لود index =====
INDEX_DIR = Path("/kaggle/working/cache/index")
embeddings = np.load(INDEX_DIR / "embeddings.npy")
with open(INDEX_DIR / "doc_ids.json") as f:
    doc_ids = json.load(f)

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
corpus = {p["doc_id"]: p["text"] for p in [json.loads(l) for l in open(PACK / "corpus.jsonl", encoding="utf-8")]}

# ===== retrieval =====
questions = [json.loads(l) for l in open(PACK / "questions_dev.jsonl", encoding="utf-8")]
q_texts = [q["question"] for q in questions]
q_embs = embedder.encode(q_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

scores = q_embs @ embeddings.T
top_idx = np.argsort(-scores, axis=1)[:, :5]
retrieved_ids = [[doc_ids[i] for i in row] for row in top_idx]

# ===== oracle =====
PROMPT = """Answer the question with the SHORTEST possible answer.
CRITICAL RULES:
- If the answer is an entity, give ONLY the entity name
- If the answer is a date, give ONLY the date
- If the answer is a place, give ONLY the place name
- If it's a yes/no question, answer exactly 'yes' or 'no'
- DO NOT write sentences
- DO NOT explain
- DO NOT start with "based on" or "the context"
- Use lowercase
- No punctuation

Context:
{context}

Question: {question}
Answer:"""

prompts = []
for q, rids in zip(questions, retrieved_ids):
    ctx = "\n\n".join(f"- {corpus[did]}" for did in rids)
    prompts.append(PROMPT.format(context=ctx, question=q["question"]))

# ===== LLM =====
llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)
sampling = SamplingParams(temperature=0, max_tokens=30)

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"dense (strict): {time.time()-t0:.1f} seconds")

# ===== save =====
OUT_DIR = Path("/kaggle/working/runs/dev/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, rids, out in zip(questions, retrieved_ids, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": rids,
        }, ensure_ascii=False) + "\n")

print("dense saved")

# ===== score =====
!python /kaggle/working/score.py /kaggle/working/runs/dev/dense/predictions.jsonl --pack /kaggle/working/data/course_pack_v1 --split dev

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [02:15<02:15, 135.57s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [02:49<00:00, 75.79s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [02:49<00:00, 84.75s/it]
(EngineCore_DP0 pid=307) 


(EngineCore_DP0 pid=307) INFO 10-14 19:14:18 [default_loader.py:267] Loading weights took 169.52 seconds
(EngineCore_DP0 pid=307) INFO 10-14 19:14:19 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 169.778846 seconds
(EngineCore_DP0 pid=307) INFO 10-14 19:14:26 [gpu_worker.py:298] Available KV cache memory: 5.92 GiB
(EngineCore_DP0 pid=307) INFO 10-14 19:14:27 [kv_cache_utils.py:1087] GPU KV cache size: 43,072 tokens
(EngineCore_DP0 pid=307) INFO 10-14 19:14:27 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.52x
(EngineCore_DP0 pid=307) WARNING 10-14 19:14:27 [cudagraph_dispatcher.py:106] cudagraph dispatching keys are not initialized. No cudagraph will be used.
(EngineCore_DP0 pid=307) INFO 10-14 19:14:27 [core.py:210] init engine (profile, create kv cache, warmup model) took 8.33 seconds
(EngineCore_DP0 pid=307) INFO 10-14 19:14:28 [__init__.py:381] Cudagraph is disabled under eager mode
INFO 10-14 19:14:28 [llm.py:306] Supported_tasks: ['ge

Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

dense (strict): 951.6 seconds
dense saved
valid: /kaggle/working/runs/dev/dense/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          62.7   [ 59.7,  65.4]
  hitall@2          28.0   [ 22.7,  33.3]
  recall@5          71.5   [ 68.6,  74.4]
  hitall@5          43.0   [ 37.7,  49.0]
  recall@10         71.5   [ 68.6,  74.4]
  hitall@10         43.0   [ 37.7,  49.0]
  mrr               97.6   [ 96.0,  98.8]
  ndcg@10           75.8   [ 73.6,  78.1]
  em                36.7   [ 31.0,  42.0]
  f1                42.6   [ 37.1,  48.1]

type: bridge_comparison (n=66)
  recall@2          46.2   [ 43.9,  48.1]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          52.3   [ 50.0,  54.9]
  hitall@5           1.5   [  0.0,   4.5]
  recall@10         52.3   [ 50.0,  54.9]
  hitall@10          1.5   [  0.0,   4.5]
  mrr               97.7   [ 94.7, 100.0]
  ndcg@10           63.8   [ 61.4,  66.1]
  em                48.5   [ 34.8,  60.6]
  f1                49.8   [ 37.2,  6

In [7]:
!zip -r runs.zip /kaggle/working/runs
from IPython.display import FileLink
FileLink("runs.zip")

  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/dev/ (stored 0%)
  adding: kaggle/working/runs/dev/dense/ (stored 0%)
  adding: kaggle/working/runs/dev/dense/predictions.jsonl (deflated 74%)


/kaggle/working/runs.zip